In [1]:
!pip install -q transformers datasets evaluate sacrebleu accelerate sentencepiece huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.4 MB/s eta 0:00:00


In [2]:
%%writefile train.py
import os
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"
os.environ["WANDB_DISABLED"] = "true"  

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict, concatenate_datasets
from transformers import (
    T5Tokenizer, 
    AutoModelForSeq2SeqLM, 
    DataCollatorForSeq2Seq, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer,
    TrainerCallback 
)
from huggingface_hub import hf_hub_download
import evaluate

MODEL_NAME = "VietAI/vit5-base"
MAX_LENGTH = 50          

# ---------------------------------------------------------
# 1. TẢI VÀ CHUẨN BỊ DỮ LIỆU
# ---------------------------------------------------------
print("1. Đang tải và chuẩn bị dữ liệu...")
train_df = pd.read_csv("https://huggingface.co/datasets/Biu3010/ViDia2Std/resolve/main/train.csv").dropna()
valid_df = pd.read_csv("https://huggingface.co/datasets/Biu3010/ViDia2Std/resolve/main/dev.csv").dropna()
test_df = pd.read_csv("https://huggingface.co/datasets/Biu3010/ViDia2Std/resolve/main/test.csv").dropna()

raw_datasets = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "validation": Dataset.from_pandas(valid_df),
    "test": Dataset.from_pandas(test_df)
})

def create_identity_pair(example):
    return {"dialect": example["standard"], "standard": example["standard"], "region": example["region"]}

identity_train = raw_datasets["train"].map(create_identity_pair)
identity_train = identity_train.select(range(len(train_df)//3))
combined_train = concatenate_datasets([raw_datasets["train"], identity_train]).shuffle(seed=42)

# ---------------------------------------------------------
# 2. TOKENIZER HACK (VƯỢT LỖI KEYERROR VÀ GIỮ TRÍ NHỚ)
# ---------------------------------------------------------
print("2. Đang cấu hình Tokenizer an toàn...")
# Tạo thư mục tạm để chứa Tokenizer
os.makedirs("local_tokenizer", exist_ok=True)
# Tải trọn bộ 3 file cấu hình sống còn của Tokenizer
hf_hub_download(repo_id=MODEL_NAME, filename="spiece.model", local_dir="local_tokenizer")
hf_hub_download(repo_id=MODEL_NAME, filename="tokenizer_config.json", local_dir="local_tokenizer")
hf_hub_download(repo_id=MODEL_NAME, filename="special_tokens_map.json", local_dir="local_tokenizer")

# Gọi Tokenizer từ thư mục nội bộ -> Né 100% mọi lỗi của HuggingFace!
tokenizer = T5Tokenizer.from_pretrained("./local_tokenizer")

def preprocess_function(examples):
    inputs =["dịch: " + text for text in examples["dialect"]]
    targets = examples["standard"]
    
    # BẮT BUỘC ĐỆM BẰNG TAY Ở ĐÂY ĐỂ TRÁNH LỖI MẢNG RỖNG
    model_inputs = tokenizer(inputs, max_length=MAX_LENGTH, padding="max_length", truncation=True)
    labels = tokenizer(targets, max_length=MAX_LENGTH, padding="max_length", truncation=True)
    
    # Tự động thay thế pad bằng -100
    model_inputs["labels"] = [[(l if l != tokenizer.pad_token_id else -100) for l in label] 
        for label in labels["input_ids"]
    ]
    return model_inputs

tokenized_train = combined_train.map(preprocess_function, batched=True)
tokenized_eval = raw_datasets["validation"].map(preprocess_function, batched=True)
tokenized_test = raw_datasets["test"].map(preprocess_function, batched=True)

# ---------------------------------------------------------
# 3. TẢI MÔ HÌNH
# ---------------------------------------------------------
print("3. Đang tải mô hình...")
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# ---------------------------------------------------------
# 4. HÀM CHẤM ĐIỂM (CÓ CAMERA DEBUG)
# ---------------------------------------------------------
sacrebleu = evaluate.load("sacrebleu")
chrf = evaluate.load("chrf")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple): preds = preds[0]
    
    # Thay -100 thành pad để decode
    preds = np.where(preds == -100, tokenizer.pad_token_id, preds)
    labels = np.where(labels == -100, tokenizer.pad_token_id, labels)
    
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    decoded_preds =[pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]
    
    # In live ra màn hình để kiểm chứng
    print(f"\n👀[DEBUG] Máy dịch: '{decoded_preds[0]}' | Đáp án: '{decoded_labels[0][0]}'\n")
    
    if all(len(p) == 0 for p in decoded_preds):
        return {"bleu": 0.0, "chrf": 0.0}
        
    bleu = sacrebleu.compute(predictions=decoded_preds, references=decoded_labels)["score"]
    chrf_score = chrf.compute(predictions=decoded_preds, references=decoded_labels)["score"]
    return {"bleu": bleu, "chrf": chrf_score}

class CSVLogCallback(TrainerCallback):
    def __init__(self): self.logs =[]
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and ("eval_bleu" in logs or "loss" in logs):
            self.logs.append(logs.copy())
            pd.DataFrame(self.logs).to_csv("training_logs.csv", index=False)

# ---------------------------------------------------------
# 5. HUẤN LUYỆN
# ---------------------------------------------------------
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="./vit5_dialect_fair",
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",
    save_only_model=True,
    report_to="none",
    
    learning_rate=5e-5,               
    fp16=False,                       
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=16,
    
    num_train_epochs=10,
    predict_with_generate=True,
    generation_num_beams=1,           
    generation_max_length=MAX_LENGTH,
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    processing_class=tokenizer,       
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[CSVLogCallback()]
)

print("🚀 4. BẮT ĐẦU HUẤN LUYỆN...")
trainer.train()

# ---------------------------------------------------------
# 6. DỰ ĐOÁN TẬP TEST VỚI BEAM SEARCH = 4
# ---------------------------------------------------------
print("🔍 5. Dự đoán tập Test với Beam Search...")
predict_args = Seq2SeqTrainingArguments(
    output_dir="./predict_temp",
    per_device_eval_batch_size=16,
    predict_with_generate=True,
    generation_num_beams=4,      
    generation_max_length=MAX_LENGTH,
    report_to="none"
)

test_trainer = Seq2SeqTrainer(
    model=model,
    args=predict_args,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

predictions, _, metrics = test_trainer.predict(tokenized_test)

if isinstance(predictions, tuple): predictions = predictions[0]

predictions = np.where(predictions == -100, tokenizer.pad_token_id, predictions)
decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

results_df = pd.DataFrame({
    "Region": raw_datasets["test"]["region"],
    "Source (Dialect)": raw_datasets["test"]["dialect"],
    "Ground Truth (Standard)": raw_datasets["test"]["standard"],
    "Model Prediction":[p.strip() for p in decoded_preds]
})
results_df.to_csv("test_predictions.csv", index=False, encoding='utf-8-sig')
print("🎉 KẾT QUẢ CUỐI CÙNG:", metrics)

Writing train.py


In [3]:
!python train.py

1. Đang tải và chuẩn bị dữ liệu...
Map: 100%|██████████████████████| 10859/10859 [00:00<00:00, 19209.36 examples/s]
2. Đang cấu hình Tokenizer an toàn...
spiece.model: 100%|██████████████████████████| 820k/820k [00:00<00:00, 1.30MB/s]
tokenizer_config.json: 2.20kB [00:00, 7.12MB/s]
special_tokens_map.json: 2.12kB [00:00, 1.08MB/s]
Map: 100%|█████████████████████████| 1603/1603 [00:00<00:00, 4477.88 examples/s]
3. Đang tải mô hình...
config.json: 100%|█████████████████████████████| 702/702 [00:00<00:00, 3.38MB/s]
pytorch_model.bin: 100%|██████████████████████| 904M/904M [00:04<00:00, 212MB/s]
Loading weights: 100%|█| 260/260 [00:00<00:00, 2736.53it/s, Materializing param=
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie